In [1]:
import numpy as np
import pandas as pd
import timesfm
import logging
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from scipy.stats import pearsonr

logging.basicConfig(
    filename='TimesFM_SSMI_Baseline_180_15_Metrics.log',
    level=logging.INFO,
    format='%(asctime)s %(levelname)s: %(message)s',
    force=True
)


def main():
    rmse_list = []
    mape_list = []
    pearson_list = []
    directional_hits = []
    try:
        # ========================
        # 1) Load SSMI data
        # ========================
        df = pd.read_csv("../../DataSets/SSMI cleaned/SSMI_cleaned.csv", parse_dates=["Date"])
        df = df.sort_values("Date").reset_index(drop=True)

        y = df["Adj Close"].values.astype(float)
        total_samples = len(y)
        logging.info(f"Total SSMI samples loaded: {total_samples}")

        # ========================
        # 2) Sliding window config
        # ========================
        context_window = 180
        forecast_horizon = 15
        step_size = 30
        num_segments = (total_samples - context_window) // step_size
        logging.info(f"Segments to evaluate: {num_segments}")

        # ========================
        # 3) Load TimesFM
        # ========================
        tfm = timesfm.TimesFm(
            hparams=timesfm.TimesFmHparams(
                backend="cpu",
                per_core_batch_size=32,
                horizon_len=forecast_horizon,
            ),
            checkpoint=timesfm.TimesFmCheckpoint(
                huggingface_repo_id="google/timesfm-1.0-200m-pytorch",
            ),
        )
        logging.info("Google TimesFM loaded successfully.")

        # ========================
        # 4) Sliding window loop (raw, no filter)
        # ========================
        for segment in range(num_segments):
            start_context = segment * step_size
            end_context = start_context + context_window
            if end_context + forecast_horizon > total_samples:
                break

            y_context = y[start_context:end_context].copy()
            y_true = y[end_context:end_context + forecast_horizon]

            point_forecast, _ = tfm.forecast([y_context], freq=[0])
            combined_pred = point_forecast[0][:forecast_horizon]

            # Directional accuracy: sign vs. prev_anchor for both actual and predicted
            prev_anchor = np.concatenate([[y[end_context - 1]], y_true[:-1]])
            actual_direction = np.sign(y_true - prev_anchor)
            pred_direction = np.sign(combined_pred - prev_anchor)
            hits = (actual_direction == pred_direction).astype(int)
            directional_hits.extend(hits.tolist())

            rmse = np.sqrt(mean_squared_error(y_true, combined_pred))
            mape = mean_absolute_percentage_error(y_true, combined_pred) * 100
            r2 = pearsonr(y_true, combined_pred).statistic ** 2

            rmse_list.append(rmse)
            mape_list.append(mape)
            pearson_list.append(r2)

            segment_dir_acc = hits.mean() * 100
            logging.info(
                f"Segment {segment+1}/{num_segments}: RMSE={rmse:.4f}, MAPE={mape:.4f}%, R\u00b2={r2:.4f}, DirAcc={segment_dir_acc:.1f}%"
            )
            print(
                f"Segment {segment+1}/{num_segments} \u2014 RMSE: {rmse:.2f} | MAPE: {mape:.2f}% | R\u00b2: {r2:.4f} | Dir Acc: {segment_dir_acc:.1f}%"
            )

        # ========================
        # 5) Save results
        # ========================
        np.savez_compressed(
            "TimesFM_SSMI_Baseline_180_15_Metrics.npz",
            rmse=np.array(rmse_list),
            mape=np.array(mape_list),
            pearson_coefficients=np.array(pearson_list),
            directional_hits=np.array(directional_hits),
            context_window=context_window,
            forecast_horizon=forecast_horizon,
            num_segments=num_segments,
        )
        logging.info("Results saved to TimesFM_SSMI_Baseline_180_15_Metrics.npz")

        # ========================
        # 6) Summary metrics
        # ========================
        total_days = len(directional_hits)
        total_hits = sum(directional_hits)
        dir_acc_pct = (total_hits / total_days) * 100 if total_days else 0.0

        print("\n--- Median Metrics for Google TimesFM on SSMI (Zero-Shot, ctx=180 / hor=15) ---")
        print(f"Median RMSE:          {np.median(rmse_list):.4f}")
        print(f"Median MAPE:          {np.median(mape_list):.4f}%")
        print(f"Median Pearson R\u00b2:    {np.median(pearson_list):.4f}")
        print(f"Directional Accuracy: {total_hits}/{total_days} days ({dir_acc_pct:.2f}%)")

    except Exception:
        logging.error("An error occurred.", exc_info=True)
        print("An error occurred. Check TimesFM_SSMI_Baseline_180_15_Metrics.log for details.")
        try:
            np.savez_compressed(
                "partial_TimesFM_SSMI_Baseline_180_15_Metrics.npz",
                rmse=np.array(rmse_list),
                mape=np.array(mape_list),
                pearson_coefficients=np.array(pearson_list),
                directional_hits=np.array(directional_hits),
            )
        except Exception:
            logging.error("Failed to save partial results.", exc_info=True)
    finally:
        logging.info("Forecasting run completed.")


if __name__ == '__main__':
    main()

 See https://github.com/google-research/timesfm/blob/master/README.md for updated APIs.
Loaded PyTorch TimesFM, likely because python version is 3.11.5 (main, Sep 11 2023, 08:31:25) [Clang 14.0.6 ].


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Segment 1/249 — RMSE: 40.91 | MAPE: 1.47% | R²: 0.1633 | Dir Acc: 60.0%
Segment 2/249 — RMSE: 12.32 | MAPE: 0.66% | R²: 0.3330 | Dir Acc: 60.0%
Segment 3/249 — RMSE: 34.23 | MAPE: 1.44% | R²: 0.3792 | Dir Acc: 53.3%
Segment 4/249 — RMSE: 22.81 | MAPE: 1.03% | R²: 0.5220 | Dir Acc: 46.7%
Segment 5/249 — RMSE: 57.17 | MAPE: 2.61% | R²: 0.6435 | Dir Acc: 40.0%
Segment 6/249 — RMSE: 25.88 | MAPE: 1.20% | R²: 0.4489 | Dir Acc: 40.0%
Segment 7/249 — RMSE: 70.04 | MAPE: 2.86% | R²: 0.8246 | Dir Acc: 33.3%
Segment 8/249 — RMSE: 53.97 | MAPE: 2.70% | R²: 0.6331 | Dir Acc: 33.3%
Segment 9/249 — RMSE: 28.71 | MAPE: 1.41% | R²: 0.5411 | Dir Acc: 66.7%
Segment 10/249 — RMSE: 59.75 | MAPE: 2.43% | R²: 0.7206 | Dir Acc: 33.3%
Segment 11/249 — RMSE: 17.97 | MAPE: 0.78% | R²: 0.5360 | Dir Acc: 60.0%
Segment 12/249 — RMSE: 69.35 | MAPE: 2.85% | R²: 0.8880 | Dir Acc: 33.3%
Segment 13/249 — RMSE: 20.57 | MAPE: 0.81% | R²: 0.0857 | Dir Acc: 73.3%
Segment 14/249 — RMSE: 36.26 | MAPE: 1.36% | R²: 0.0548 | Di